In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Circle
from scipy.interpolate import interp1d
import pandas as pd
import sncosmo
from scipy import stats
from scipy.special import expit
from nested_pandas import read_parquet
from joblib import Parallel, delayed
import cloudpickle as pickle
from regions import RectangleSkyRegion
from astropy.coordinates import SkyCoord
import astropy.units as u
import glob
from astropy.io import fits

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lightcurvelynx.obstable.ztf_obstable import ZTFObsTable, _ztfcam_ccd_gain, _ztfcam_readout_noise
from lightcurvelynx.astro_utils.mag_flux import mag2flux
import sqlite3

from lightcurvelynx.obstable.ztf_obstable import ZTFObsTable
from lightcurvelynx.obstable.fake_obs_table import FakeObsTable
from lightcurvelynx.astro_utils.passbands import PassbandGroup
from lightcurvelynx.astro_utils.pzflow_node import PZFlowNode
from lightcurvelynx.math_nodes.scipy_random import SamplePDF
from lightcurvelynx.math_nodes.np_random import NumpyRandomFunc
from lightcurvelynx.simulate import simulate_lightcurves
from lightcurvelynx.models.sncomso_models import SncosmoWrapperModel
from lightcurvelynx.models.snia_host import SNIaHost
from lightcurvelynx.utils.plotting import plot_lightcurves
from lightcurvelynx.math_nodes.ra_dec_sampler import ObsTableUniformRADECSampler
from lightcurvelynx.astro_utils.dustmap import DustmapWrapper,SFDMap
from lightcurvelynx.effects.extinction import ExtinctionEffect
from lightcurvelynx.astro_utils.mag_flux import mag2flux,flux2mag
from lightcurvelynx.astro_utils.detector_footprint import DetectorFootprint

from lightcurvelynx.graph_state import GraphState
from lightcurvelynx.simulate import compute_single_noise_free_lightcurve
from lightcurvelynx.astro_utils.noise_model import poisson_bandflux_std

from lightcurvelynx import _LIGHTCURVELYNX_BASE_DATA_DIR
from lightcurvelynx.consts import GAUSS_EFF_AREA2FWHM_SQ

from lightcurvelynx.validation.lcfit import fit_single_lc

from utils.plotting_utils import convert_flux_to_njy, plot_coverage_map

In [ ]:
globalhostdata = pd.read_csv('ztfsniadr2/tables/globalhost_data.csv')
localhostdata = pd.read_csv('ztfsniadr2/tables/localhost_data.csv')
sndata = pd.read_csv('ztfsniadr2/tables/snia_data.csv')
data = pd.merge(sndata,globalhostdata,on='ztfname')

In [ ]:
lcdata = read_parquet('data/ztfsniadr2.parquet')

In [ ]:
list2 = pd.read_csv('/Users/mi/Downloads/ztfnames',header=None)[0].to_list()

In [ ]:
list1 = data.iau_name.to_list()

In [ ]:
# randomly pick an SN in the data release to simulate
# ZTF20abegoix - this one looks good for paper
# ZTF19aazlsfj - also good
random_sn = data.dropna().sample()
# random_sn = data.loc[data.iau_name == "2020jny"]
# random_sn = data.loc[data.ztfname == "ZTF19aazlsfj"]
random_sn

In [ ]:
random_lc = lcdata.loc[lcdata["ztfname"] == random_sn.ztfname.values[0]]
random_lc['lc'].iloc[0]

In [ ]:
obs_log = pd.read_parquet('data/ztf_observing_log_combined_w_metadata.parquet')
colmap = {"ra":"ra",
          "dec":"dec",
          "time":"mjd",
          "zp":"zp_nJy",
          "filter":"filter",
          "sky":"sky_adu",
         }

#ztf ccd size 6144 × 6160 pixel * 16
pixel_scale = 1.01 #arcsec/pixel
center = SkyCoord(ra=0.0, dec=0.0, unit="deg", frame="icrs")
rect_region = RectangleSkyRegion(center=center, width=4*6144.* pixel_scale * u.arcsec, 
                                 height=4*6160.* pixel_scale * u.arcsec, angle=0.0 * u.deg)
ztf_fp = DetectorFootprint(rect_region, pixel_scale=pixel_scale)

ztf_obstable = ZTFObsTable(obs_log,colmap=colmap,detector_footprint=ztf_fp)

In [ ]:
ra, dec = random_sn.ra.values[0], random_sn.dec.values[0]
idx = ztf_obstable.range_search(ra,dec)
table = ztf_obstable._table.iloc[idx]
table

In [ ]:
obs_log_allccd = pd.read_parquet('ztfsniadr2/tables/observing_logs.parquet')
obs_log_allccd = obs_log_allccd[obs_log_allccd["expid"].isin(table.expid)]
obs_log_allccd

In [ ]:
df1 = table
df2 = random_lc['lc'].iloc[0]
df3 = obs_log_allccd

In [ ]:
# match the rcid for that sn
dflist = []
for i,row in df2.drop_duplicates(['field_id','rcid']).iterrows():
    df = df3.loc[(df3.fieldid == row['field_id']) & (df3.rcid == row['rcid'])]
    dflist.append(df)
obs_log = pd.concat(dflist)
obs_log

In [ ]:
con = sqlite3.connect("data/ztf_metadata_latest.db")
sql_query = "SELECT * FROM exposures"
metadata_table = pd.read_sql_query(sql_query, con)
metadata_table = metadata_table.replace("", np.nan)
metadata_table = metadata_table.dropna(subset=["fwhm"])
metadata_table.columns

In [ ]:
obs_log["filter"] = obs_log.apply(lambda row: row["band"][-1],axis=1)
obs_log = pd.merge(obs_log, metadata_table[["expid","filter","exptime","fwhm","obsdate","scibckgnd","ra","dec","maglim"]],on=["filter","expid"])
gain = _ztfcam_ccd_gain
obs_log["zp_nJy"] = mag2flux(obs_log["zp"].values + 2.5*np.log10(gain))
obs_log = obs_log.rename(columns={"zp":"zp_abmag"})
def compute_sky(row):
    gain = _ztfcam_ccd_gain
    nea = GAUSS_EFF_AREA2FWHM_SQ * (row["fwhm"]) ** 2
    flux = np.power(10., -0.4*(row['maglimit'] - row['zp_abmag'])) * gain
    sky = (flux**2 / 25 - flux - _ztfcam_readout_noise**2 * nea) / nea / gain
    return sky    
obs_log["sky_adu"] = obs_log.apply(compute_sky,axis=1)
obs_log

In [ ]:
colmap = {"ra":"ra",
          "dec":"dec",
          "time":"mjd",
          "zp":"zp_nJy",
          "filter":"filter",
          "sky":"sky_adu",
         }

#ztf ccd size 6144 × 6160 pixel * 16
pixel_scale = 1.01 #arcsec/pixel
center = SkyCoord(ra=0.0, dec=0.0, unit="deg", frame="icrs")
rect_region = RectangleSkyRegion(center=center, width=4*6144.* pixel_scale * u.arcsec, 
                                 height=4*6160.* pixel_scale * u.arcsec, angle=0.0 * u.deg)
ztf_fp = DetectorFootprint(rect_region, pixel_scale=pixel_scale)

ztf_obstable = ZTFObsTable(obs_log,colmap=colmap,detector_footprint=ztf_fp)
ztf_obstable.survey_values["zp_err_mag"] = 0.001

t_min, t_max = ztf_obstable.time_bounds()
print(f"Loaded OpSim with {len(ztf_obstable)} rows and times [{t_min}, {t_max}]")

# sky_coverage = ztf_obstable.estimate_coverage(use_footprint=True)
# print(f"The total sky coverage is {sky_coverage} square degrees")

passband_group = PassbandGroup.from_preset(preset="ZTF", filters=["g", "r", "i"])
print(f"Loaded Passbands: {passband_group}")

In [ ]:
H0 = 70.0
Omega_m = 0.3

host = SNIaHost(
    ra = random_sn.ra_host,
    dec = random_sn.dec,
    hostmass= random_sn.mass,
    redshift=random_sn.redshift,
    node_label="host",
)

In [ ]:
sncosmo_modelname = "salt3"
source = SncosmoWrapperModel(
    sncosmo_modelname,
    t0=random_sn.t0.values[0],
    x0=random_sn.x0.values[0],
    x1=random_sn.x1.values[0],
    c=random_sn.c.values[0],
    ra=random_sn.ra.values[0],
    dec=random_sn.dec.values[0],
    redshift=random_sn.redshift.values[0],
    node_label="source",
)
    
mwextinction = SFDMap(
    ra=source.ra,
    dec=source.dec,
    node_label="mwext",
)

# Create an extinction effect using the EBVs from that dust map.
ext_effect = ExtinctionEffect(extinction_model="F99", frame='observer', ebv=mwextinction, Rv=3.1)
source.add_effect(ext_effect)


In [ ]:
nsntotal = 100
lightcurves = simulate_lightcurves(source, int(nsntotal), ztf_obstable, passband_group, 
                                   obstable_save_cols=["expid","zp_nJy","scibckgnd","skynoise",
                                                       "fwhm","maglimit","maglim","sky_adu"])
lightcurves

In [ ]:
plot_coverage_map(ztf_obstable,lightcurves,plot_na_location=False,plot_all_location=True)

In [ ]:
lightcurves = lightcurves.rename(columns={"lightcurve":"lc"})
lightcurves = lightcurves.dropna(subset="lc")

In [ ]:
lightcurves['lc.snr'] = lightcurves['lc.flux']/lightcurves['lc.fluxerr']
detection_snr_thres = 5.
lightcurves['lc.detection_flag'] = lightcurves['lc.snr'] > detection_snr_thres
# drop saturation
lightcurves_after_drop_sat = lightcurves.query("lc.is_saturated==False").dropna()
# drop non detection
lightcurves_after_detection = lightcurves_after_drop_sat.query("lc.detection_flag == True").dropna()

In [ ]:
def filter_flags(lc_flag, flags_to_exclude=[], flags_to_include=[]):
    pass_filter = True
    if len(flags_to_include)>0:
        pass_filter &= np.all([lc_flag & flag != 0 for flag in flags_to_include])
    if len(flags_to_exclude)>0:
        pass_filter &= np.all([lc_flag & flag == 0 for flag in flags_to_exclude])
    return pass_filter

In [ ]:
sncosmo_modelname = "salt3"
random_ids = lightcurves.id.sample(1).values
colormap = {'g':'g',
            'r':'r',
            'i':'purple',}

ax = plt.subplot()
        
# lc_all = lightcurves_after_detection
lc_all = lightcurves.dropna(subset=["lc"])

ax = plot_lightcurves(
    fluxes=lc_all["lc.flux"],
    times=lc_all["lc.mjd"],
    filters=lc_all["lc.filter"],
    colormap=colormap,
    underlying_model=None,#noise_free_lcs,
    alpha = 0.007,
    ax = ax,
)


plt.ylabel('Flux (nJy)')

lc_plot = random_lc
# lc_plot["lc.pass_flag_filter"] = lc_plot["lc.flag"].apply(filter_flags,flags_to_exclude=[1,2,4,8,16,32],flags_to_include=[])
# lc_plot = lc_plot.query("lc.pass_flag_filter == True")

#plot the data
lc_plot["lc.flux"], lc_plot["lc.flux_err"] = convert_flux_to_njy(lc_plot["lc.flux"],lc_plot["lc.flux_err"],zp=30.)

plot_lightcurves(
    fluxes=lc_plot["lc.flux"],
    times=lc_plot["lc.mjd"],
    fluxerrs=lc_plot["lc.flux_err"],
    filters=[x[-1] for x in lc_plot["lc.filter"]],
    colormap=colormap,
    ax = ax,
    marker = "o",
    markeredgecolor = "k",
)

t_min, t_max = -2 + random_sn.t0.values[0] - 20*(1+random_sn.redshift.values[0]), 2 + random_sn.t0.values[0] + 50*(1+random_sn.redshift.values[0])
plt.xlim((t_min,t_max))
plt.gca().get_legend().remove()
# plt.ylim((-5e4,mag2flux(17)))
plt.tight_layout()

# plt.savefig("paper_figs/example_lc.png")

In [ ]:
t_min = lc_plot["lc.mjd"].min()
t_max = lc_plot["lc.mjd"].max()

In [ ]:
lynx_lc = lc_all.query(f"lc.mjd > {t_min} & lc.mjd < {t_max}")["lc"].iloc[0]
ztf_lc = lc_plot.query(f"lc.mjd > {t_min} & lc.mjd < {t_max}")["lc"].iloc[0]

In [ ]:
bins = np.arange(t_min,t_max,1)
plt.hist(lynx_lc.mjd,bins=bins,alpha=0.5)
plt.hist(ztf_lc.mjd,bins=bins,alpha=0.5)
plt.show()

In [ ]:
from astropy.time import Time
t = Time([t_min,t_max], format='mjd', scale='utc')
t.to_datetime()

In [ ]:
ztf_lc = lc_plot.query(f"lc.mjd > {t_min} & lc.mjd < {t_max}")["lc"].iloc[0]

for i in range(0,len(lightcurves)):
    lynx_lc = lc_all.query(f"lc.mjd > {t_min} & lc.mjd < {t_max}")["lc"].iloc[i]
    
    merged = pd.merge_asof(
        lynx_lc.sort_values('mjd'),
        ztf_lc.sort_values('mjd'),
        on='mjd',
        direction='nearest',
        tolerance=0.01,
        suffixes = ('_lynx','_ztf'),
    )
    merged = merged.dropna().loc[merged.dropna().detection_flag]
    plt.plot(merged.mjd,merged.flux_lynx/merged.flux_ztf,'o',alpha=0.02,color='b')
plt.ylim((0,2))
plt.xlabel('MJD')
plt.ylabel("flux_sim/flux_data")

In [ ]:
ztf_lc = lc_plot.query(f"lc.mjd > {t_min} & lc.mjd < {t_max}")["lc"].iloc[0]

for i in range(0,len(lightcurves)):
    lynx_lc = lc_all.query(f"lc.mjd > {t_min} & lc.mjd < {t_max}")["lc"].iloc[i]
    
    merged = pd.merge_asof(
        lynx_lc.sort_values('mjd'),
        ztf_lc.sort_values('mjd'),
        on='mjd',
        direction='nearest',
        tolerance=0.01,
        suffixes = ('_lynx','_ztf'),
    )
    merged = merged.dropna().loc[merged.dropna().detection_flag]
    plt.plot(merged.mjd,merged.fluxerr/merged.flux_err,'o',color='b',alpha=0.02)
plt.ylim((0.,2))
plt.xlabel('MJD')
plt.ylabel("fluxerr_sim/fluxerr_data")

In [ ]:
ztf_lc = lc_plot.query(f"lc.mjd > {t_min} & lc.mjd < {t_max}")["lc"].iloc[0]

for i in range(0,len(lightcurves)):
    lynx_lc = lc_all.query(f"lc.mjd > {t_min} & lc.mjd < {t_max}")["lc"].iloc[i]
    
    merged = pd.merge_asof(
        lynx_lc.sort_values('mjd'),
        ztf_lc.sort_values('mjd'),
        on='mjd',
        direction='nearest',
        tolerance=0.01,
        suffixes = ('_lynx','_ztf'),
    )
    merged = merged.dropna().loc[merged.dropna().detection_flag]
    plt.plot(merged.mjd,(merged.flux_lynx/merged.fluxerr)/(merged.flux_ztf/merged.flux_err),'o',color='b',alpha=0.02)
plt.ylim((0,5))
plt.xlabel('MJD')
plt.ylabel("snr_sim/snr_data")

In [ ]:
(merged.fluxerr).hist(bins=np.linspace(0,1e4,20),alpha=0.5,label='sim')
(merged.flux_err).hist(bins=np.linspace(0,1e4,20),alpha=0.5,label='data')
plt.legend()